# 01 - Sentinel-2 download

Builds the 2025 Sentinel-2 median mosaic of the Monterrey Metropolitan Area (ZMM) in Google Earth Engine and exports it to Google Drive:

- **Tiles for prediction:** the ZMM bounding box is split into 3x3 quadrants (Q11-Q33) and each quadrant into a 15x15 grid, one export per tile.
- **Full mosaic:** exported once, then merged and split into the same 9 quadrants locally (used for NDVI maps and figures).

Bands: B2, B3, B4, B8, B8A, B11, B12 at 10 m, EPSG:4326.

It also builds the 2025 land surface temperature (LST) map of the ZMM from Landsat-8, used in `04_ageb_indicators.ipynb`.

Note on the training tiles: the labeled tiles (C1-C9, `Sentinel_Monterrey_15x15_Rxx_Cxx.tif`) come from a first version of this grid. It covered a smaller area and its tiles were not square (about 199x187 px, 2 km). The grid was then rebuilt as Q11-Q33 (about 292x291 px, 3 km) and the model, already trained on the C tiles, was applied to the Q tiles. Same collection, filters and bands, so features are comparable, but the C and Q tiles are different exports.

ROI definition and 3x3 grid

In [ ]:
import ee
from config import geeProject, boundaryAsset

ee.Initialize(project=geeProject)

# ZMM boundary uploaded to GEE as an asset
roiFc=ee.FeatureCollection(boundaryAsset)
roiTotal=roiFc.geometry()

# Bounding box that contains the ZMM
bounds=roiTotal.bounds().getInfo()['coordinates'][0]
lonMin=bounds[0][0]
latMin=bounds[0][1]
lonMax=bounds[2][0]
latMax=bounds[2][1]

# Size of each quadrant
quadLon=(lonMax-lonMin)/3
quadLat=(latMax-latMin)/3

General mosaic

In [ ]:
# Sentinel-2 collection for 2025
sentinelRaw=(
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roiTotal)
    .filterDate('2025-01-01', '2025-11-27')
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 5))  # less than 5% clouds
)

# Mask with the Scene Classification Layer
# 3 = cloud shadow, 8 = cloud medium prob., 9 = cloud high prob., 10 = cirrus
def maskClouds(image):
    scl=image.select('SCL')
    cloudMask=scl.remap([3, 8, 9, 10], [0, 0, 0, 0], 1)
    return image.updateMask(cloudMask)

sentinelClean=sentinelRaw.map(maskClouds)

# Median mosaic
sentinel=sentinelClean.select(['B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12'])
sentinelFinal=sentinel.median()
imageZmm=sentinelFinal.clip(roiTotal)

Function to split a quadrant and create the GEE export tasks

In [ ]:
def exportQuadrant(qRow, qCol, image, subRow, subCol, prefix, folderName):
    '''
    Exports the tiles of one quadrant according to the number of subdivisions
    '''

    qLonMin=lonMin+(qCol-1)*quadLon
    qLonMax=lonMin+qCol*quadLon
    qLatMin=latMin+(qRow-1)*quadLat
    qLatMax=latMin+qRow*quadLat

    # Size of the subdivisions
    subLon=(qLonMax-qLonMin)/subCol
    subLat=(qLatMax-qLatMin)/subRow

    quadId=f'Q{qRow}{qCol}'
    totalTiles=subRow*subCol
    tasksStarted=0

    print(f'Quadrant {quadId}: lon [{qLonMin}, {qLonMax}], lat [{qLatMin}, {qLatMax}]')
    print(f'  Grid: {subRow} rows x {subCol} cols ({totalTiles} tiles)')

    for i in range(subRow):
        for j in range(subCol):
            cLonMin=qLonMin+j*subLon
            cLonMax=qLonMin+(j+1)*subLon
            cLatMin=qLatMin+i*subLat
            cLatMax=qLatMin+(i+1)*subLat

            cell=ee.Geometry.BBox(cLonMin, cLatMin, cLonMax, cLatMax)

            tileId=f'{quadId}_R{i+1:02d}_C{j+1:02d}'

            task=ee.batch.Export.image.toDrive(
                image=image,
                description=f'Export_{tileId}',
                folder=folderName,
                fileNamePrefix=f'{prefix}_{tileId}',
                region=cell,
                scale=10,
                crs='EPSG:4326',
                maxPixels=1e10
            )

            task.start()
            tasksStarted+=1

    print(f'Quadrant {quadId}: {tasksStarted} tasks started')

Export of the tiles by quadrant. Each quadrant goes to its own Drive folder (later copied to `01_raw/SENTINEL2/Qxx`)

In [ ]:
quadrants=[(1,1),(1,2),(1,3),
           (2,1),(2,2),(2,3),
           (3,1),(3,2),(3,3)]

for qRow,qCol in quadrants:
    exportQuadrant(
        qRow=qRow,
        qCol=qCol,
        image=imageZmm,
        subRow=15,
        subCol=15,
        prefix='S2_ZMM_2025',
        folderName=f'S2_ZMM_2025_Q{qRow}{qCol}'
    )

Export of the full mosaic. GEE splits large exports into several files (`S2_ZMM_2025_mosaic_full-*.tif`), they are merged below

In [ ]:
task=ee.batch.Export.image.toDrive(
    image=imageZmm,
    description='Export_mosaic_full',
    folder='S2_ZMM_2025_Full',
    fileNamePrefix='S2_ZMM_2025_mosaic_full',
    region=roiTotal.bounds(),
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e13
)
task.start()

## Land surface temperature (Landsat-8)

The original LST script was written in the GEE Code Editor and was not saved. This cell is rewritten from the parameters documented in the project notes: Landsat-8 Collection 2 Level 2, band `ST_B10`, scenes with less than 50% clouds, clouds and shadows masked with `QA_PIXEL`, conversion to deg C and median of 2025, exported at 30 m.

It is exported with a different name (`LST_ZMM_2025_rebuilt`) and compared with the original raster at the end of this notebook.

In [ ]:
# Landsat-8, Collection 2, Level 2 (includes surface temperature)
landsatRaw=(
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(roiTotal)
    .filterDate('2025-01-01', '2025-11-27')
    .filter(ee.Filter.lt('CLOUD_COVER', 50))
)

# Clouds and shadows from QA_PIXEL: bit 3 = cloud, bit 4 = cloud shadow
def maskLandsat(image):
    qa=image.select('QA_PIXEL')
    cloud=qa.bitwiseAnd(1<<3).neq(0)
    shadow=qa.bitwiseAnd(1<<4).neq(0)
    return image.updateMask(cloud.Or(shadow).Not())

# ST_B10 to deg C with the Landsat scale factor and offset
def toCelsius(image):
    lstK=image.select('ST_B10').multiply(0.00341802).add(149.0)
    return lstK.subtract(273.15).rename('LST_C')

lstZmm=landsatRaw.map(maskLandsat).map(toCelsius).median().clip(roiTotal)

task=ee.batch.Export.image.toDrive(
    image=lstZmm,
    description='Export_LST_ZMM_2025',
    folder='LST_ZMM_2025',
    fileNamePrefix='LST_ZMM_2025_rebuilt',
    region=roiTotal.bounds(),
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e10
)
task.start()

# Image processing
Merge the files of the full mosaic into a single raster

In [ ]:
import rasterio
from rasterio.merge import merge
from config import s2FullFolder

tifPaths=sorted(s2FullFolder.glob('S2_ZMM_2025_mosaic_full-*.tif'))
print('TIF found:',len(tifPaths))
for p in tifPaths:
    with rasterio.open(p) as src:
        print(p.name,'->',src.width,'x',src.height)

mergedPath=s2FullFolder/'S2_ZMM_2025_mosaic_full_merged.tif'

# dst_path writes the result directly to disk (the mosaic is ~9.5 GB)
srcFiles=[rasterio.open(p) for p in tifPaths]
merge(srcFiles,dst_path=mergedPath)

for src in srcFiles:
    src.close()

print('Saved:',mergedPath)

Split the full mosaic into 9 quadrants

In [ ]:
from rasterio.windows import Window

with rasterio.open(mergedPath) as src:
    width=src.width
    height=src.height

    nCols=3
    nRows=3

    tileW=width//nCols
    tileH=height//nRows

    for i in range(nRows):
        for j in range(nCols):
            colOff=j*tileW
            rowOff=i*tileH

            # the last row/column keeps the remaining pixels
            w=tileW if j<nCols-1 else (width-colOff)
            h=tileH if i<nRows-1 else (height-rowOff)

            window=Window(colOff, rowOff, w, h)

            profile=src.profile.copy()
            profile.update(
                width=w,
                height=h,
                transform=rasterio.windows.transform(window, src.transform)
            )

            outName=s2FullFolder/'quadrants'/f'S2_ZMM_2025_mosaic_Q{i+1}{j+1}.tif'
            outName.parent.mkdir(exist_ok=True)
            with rasterio.open(outName,'w',**profile) as dst:
                dst.write(src.read(window=window))

            print('Saved:',outName)

Figure of the ZMM with the 9 quadrants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

downscale=10   # read the mosaic at 1/10 resolution, the full one does not fit in memory

with rasterio.open(mergedPath) as src:
    H=src.height//downscale
    W=src.width//downscale
    rgb=src.read([3,2,1],out_shape=(3,H,W)).astype('float32')   # B4, B3, B2

# Simple stretch with global percentiles, ignoring the 0 background
vals=rgb[rgb>0]
p2,p98=np.percentile(vals,(2,98))
rgb=np.clip((rgb-p2)/(p98-p2+1e-6),0,1)
rgb=np.transpose(rgb,(1,2,0))

fig,ax=plt.subplots(figsize=(10,10))
ax.imshow(rgb)

# Quadrant lines
for j in range(1,3):
    ax.axvline(x=j*W/3,color='white',linestyle='--',linewidth=1)
for i in range(1,3):
    ax.axhline(y=i*H/3,color='white',linestyle='--',linewidth=1)

ax.set_xticks([])
ax.set_yticks([])
ax.set_title('ZMM Sentinel-2 2025 mosaic (9 quadrants) - true color')
plt.tight_layout()

outFig=s2FullFolder/'S2_ZMM_2025_mosaic_full_9quadrants.png'
plt.savefig(outFig,dpi=300,bbox_inches='tight')
plt.show()
print('Saved:',outFig)

## Check of the rebuilt LST
Download `LST_ZMM_2025_rebuilt.tif` from Drive to `data/raw/lst/` and compare it with the original raster used in the analysis. This cell can be run on its own.

In [ ]:
import numpy as np
import rasterio
from config import lstFile

rebuiltPath=lstFile.parent/'LST_ZMM_2025_rebuilt.tif'

with rasterio.open(lstFile) as src:
    lstOriginal=src.read(1)
with rasterio.open(rebuiltPath) as src:
    lstRebuilt=src.read(1)

print('Shape original:',lstOriginal.shape,' rebuilt:',lstRebuilt.shape)

if lstOriginal.shape==lstRebuilt.shape:
    valid=np.isfinite(lstOriginal)&np.isfinite(lstRebuilt)
    diff=lstRebuilt[valid]-lstOriginal[valid]
    print(f'Valid pixels: {valid.sum()}')
    print(f'Correlation: {np.corrcoef(lstOriginal[valid],lstRebuilt[valid])[0,1]:.4f}')
    print(f'Mean abs difference: {np.abs(diff).mean():.3f} deg C, max: {np.abs(diff).max():.3f} deg C')